# 🏎️ F1 BI — Pipeline ETL
## Proyecto Final · Módulo 4: Inteligencia de Negocios y SQL Avanzado

**Dataset:** Ergast F1 Database (1950–2024)  
**Destino:** Aurora PostgreSQL — Schema `f1_dw`  
**Grano:** Un registro por piloto por carrera

| Fase | Descripción |
|---|---|
| Extract | Lee los 14 CSVs del dataset Ergast |
| Transform | Construye dimensiones y fact table |
| Load | Upsert idempotente en Aurora |
| Validate | Conteos, integridad referencial, nulos |

## 0. Instalación de dependencias

In [ ]:
# Ejecutar solo la primera vez en Google Colab
!pip install pandas sqlalchemy psycopg2-binary python-dotenv -q

## 1. Imports y logging

In [ ]:
import os, logging
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger('f1_etl')
logger.info('Imports cargados correctamente')

## 2. Conexión a Aurora PostgreSQL

In [ ]:
# En Google Colab descomenta las siguientes líneas:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/proyecto-final/datasets'

DATA_PATH = './datasets'  # Ajusta según tu entorno
SCHEMA    = 'f1_dw'

load_dotenv()
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT','5432')}/{os.getenv('DB_NAME')}",
    pool_pre_ping=True
)
with engine.connect() as conn:
    v = conn.execute(text('SELECT version()')).scalar()
    logger.info(f'Conexion exitosa: {v[:60]}...')

## 3. EXTRACT — Lectura de CSVs

In [ ]:
def extract(data_path):
    archivos = {
        'races':'races.csv','results':'results.csv','drivers':'drivers.csv',
        'constructors':'constructors.csv','circuits':'circuits.csv',
        'pit_stops':'pit_stops.csv','lap_times':'lap_times.csv',
        'qualifying':'qualifying.csv','driver_standings':'driver_standings.csv',
        'constructor_standings':'constructor_standings.csv',
        'constructor_results':'constructor_results.csv','seasons':'seasons.csv',
        'status':'status.csv','sprint_results':'sprint_results.csv',
    }
    raw = {}
    for nombre, archivo in archivos.items():
        ruta = os.path.join(data_path, archivo)
        if not os.path.exists(ruta):
            logger.warning(f'  No encontrado (omitido): {ruta}')
            continue
        df = pd.read_csv(ruta, na_values=['\\N','NA','','NULL'])
        raw[nombre] = df
        logger.info(f'  {archivo:<40} -> {len(df):>8,} filas')
    return raw

raw = extract(DATA_PATH)

## 4. TRANSFORM — Dimensiones

In [ ]:
def asignar_era(y):
    if pd.isna(y) or y<=1966: return 'Aspiracion natural 1950-66'
    elif y<=1976: return 'Era DFV 1967-76'
    elif y<=1988: return 'Era turbo 1977-88'
    elif y<=2005: return 'NA moderno 1989-2005'
    elif y<=2013: return 'Era V8 2006-13'
    else:         return 'Era hibrida 2014+'

def transform_dim_piloto(raw):
    df = raw['drivers'].copy()
    df['nombre_completo'] = (df['forename'].fillna('')+' '+df['surname'].fillna('')).str.strip()
    df['fecha_nacimiento'] = pd.to_datetime(df['dob'], errors='coerce')
    dim = df.rename(columns={'driverId':'driver_id','driverRef':'driver_ref',
        'number':'numero_permanente','code':'codigo_piloto','nationality':'nacionalidad'})
    dim = dim[['driver_id','driver_ref','numero_permanente','codigo_piloto',
               'nombre_completo','fecha_nacimiento','nacionalidad']]
    dim = dim.sort_values('driver_id').reset_index(drop=True)
    dim.insert(0,'piloto_sk',dim.index+1)
    logger.info(f'dim_piloto:      {len(dim):,} filas'); return dim

def transform_dim_constructor(raw):
    df = raw['constructors'].copy()
    if 'constructor_results' in raw and 'races' in raw:
        cr = raw['constructor_results'].merge(raw['races'][['raceId','year']],on='raceId',how='left')
        em = cr.groupby('constructorId')['year'].median().reset_index()
        em.columns=['constructorId','my']
        em['era_f1'] = em['my'].apply(asignar_era)
        df = df.merge(em[['constructorId','era_f1']],on='constructorId',how='left')
    else:
        df['era_f1'] = 'Desconocido'
    dim = df.rename(columns={'constructorId':'constructor_id','constructorRef':'constructor_ref',
        'name':'nombre','nationality':'nacionalidad'})
    dim = dim[['constructor_id','constructor_ref','nombre','nacionalidad','era_f1']]
    dim = dim.sort_values('constructor_id').reset_index(drop=True)
    dim.insert(0,'constructor_sk',dim.index+1)
    logger.info(f'dim_constructor: {len(dim):,} filas'); return dim

def transform_dim_circuito(raw):
    df = raw['circuits'].copy()
    dim = df.rename(columns={'circuitId':'circuit_id','circuitRef':'circuit_ref',
        'name':'nombre','location':'localidad','country':'pais','lat':'latitud','lng':'longitud'})
    dim = dim[['circuit_id','circuit_ref','nombre','localidad','pais','latitud','longitud']]
    dim['latitud']  = pd.to_numeric(dim['latitud'],  errors='coerce')
    dim['longitud'] = pd.to_numeric(dim['longitud'], errors='coerce')
    dim = dim.sort_values('circuit_id').reset_index(drop=True)
    dim.insert(0,'circuito_sk',dim.index+1)
    logger.info(f'dim_circuito:    {len(dim):,} filas'); return dim

def transform_dim_tiempo(raw):
    df = raw['races'].copy()
    df['era_f1'] = df['year'].apply(asignar_era)
    df['fecha']  = pd.to_datetime(df['date'], errors='coerce')
    dim = df.rename(columns={'raceId':'race_id','year':'anio','name':'nombre_gp','round':'numero_ronda'})
    dim = dim[['race_id','fecha','anio','nombre_gp','numero_ronda','era_f1']].copy()
    dim['temporada'] = dim['anio']
    dim = dim.sort_values('race_id').reset_index(drop=True)
    dim.insert(0,'tiempo_sk',dim.index+1)
    logger.info(f'dim_tiempo:      {len(dim):,} filas'); return dim

def transform_dim_estado(raw):
    df = raw['status'].copy()
    mec=['Engine','Gearbox','Transmission','Clutch','Hydraulics','Electrical','Brakes',
         'Suspension','Overheating','Mechanical','Turbo','ERS','Battery','Tyre','Puncture',
         'Wheel','Driveshaft','Steering','Power Unit','Pneumatics','Oil pressure']
    acc=['Accident','Collision','Spun off','Collision damage','Fatal accident']
    des=['Disqualified','Excluded','Not classified','Did not qualify','107% rule','Underweight']
    def cat(s):
        if s=='Finished' or '+' in str(s) or 'Lap' in str(s): return 'Finalizado'
        elif any(m.lower() in s.lower() for m in mec): return 'Abandono mecanico'
        elif any(a.lower() in s.lower() for a in acc): return 'Accidente'
        elif any(d.lower() in s.lower() for d in des): return 'Descalificado'
        else: return 'Otro'
    df['categoria'] = df['status'].apply(cat)
    dim = df.rename(columns={'statusId':'status_id','status':'descripcion'})
    dim = dim[['status_id','descripcion','categoria']]
    dim = dim.sort_values('status_id').reset_index(drop=True)
    dim.insert(0,'estado_sk',dim.index+1)
    logger.info(f'dim_estado:      {len(dim):,} filas')
    logger.info(f'Categorias:\n{dim["categoria"].value_counts().to_string()}')
    return dim

dim_piloto      = transform_dim_piloto(raw)
dim_constructor = transform_dim_constructor(raw)
dim_circuito    = transform_dim_circuito(raw)
dim_tiempo      = transform_dim_tiempo(raw)
dim_estado      = transform_dim_estado(raw)

## 5. TRANSFORM — Fact Table

In [ ]:
def transform_fact(raw, dim_piloto, dim_constructor, dim_circuito, dim_tiempo, dim_estado):
    results = raw['results'].copy()
    races   = raw['races'][['raceId','circuitId']].copy()
    if 'pit_stops' in raw:
        pc = raw['pit_stops'].groupby(['raceId','driverId']).size().reset_index(name='num_pitstops')
        results = results.merge(pc, on=['raceId','driverId'], how='left')
        results['num_pitstops'] = results['num_pitstops'].fillna(0).astype(int)
    else:
        results['num_pitstops'] = 0
    results = results.merge(races, on='raceId', how='left')
    results = results.merge(dim_piloto[['piloto_sk','driver_id']],             left_on='driverId',      right_on='driver_id',      how='left')
    results = results.merge(dim_constructor[['constructor_sk','constructor_id']],left_on='constructorId', right_on='constructor_id', how='left')
    results = results.merge(dim_circuito[['circuito_sk','circuit_id']],         left_on='circuitId',     right_on='circuit_id',     how='left')
    results = results.merge(dim_tiempo[['tiempo_sk','race_id']],                left_on='raceId',        right_on='race_id',        how='left')
    results = results.merge(dim_estado[['estado_sk','status_id']],              left_on='statusId',      right_on='status_id',      how='left')
    results['posicion_salida']     = pd.to_numeric(results['grid'],          errors='coerce')
    results['posicion_final']      = pd.to_numeric(results['positionOrder'], errors='coerce')
    results['puntos']              = pd.to_numeric(results['points'],        errors='coerce').fillna(0)
    results['vueltas_completadas'] = pd.to_numeric(results['laps'],          errors='coerce').fillna(0).astype(int)
    results['tiempo_total_ms']     = pd.to_numeric(results['milliseconds'],  errors='coerce')
    results['delta_posicion']      = results['posicion_salida'] - results['posicion_final']
    fin = dim_estado[dim_estado['categoria']=='Finalizado']['estado_sk'].tolist()
    results['es_abandono'] = ~results['estado_sk'].isin(fin)
    fact = results[['piloto_sk','constructor_sk','circuito_sk','tiempo_sk','estado_sk',
        'posicion_salida','posicion_final','puntos','vueltas_completadas',
        'num_pitstops','tiempo_total_ms','delta_posicion','es_abandono']].copy()
    fact = fact.dropna(subset=['piloto_sk','constructor_sk','circuito_sk','tiempo_sk','estado_sk'])
    for col in ['piloto_sk','constructor_sk','circuito_sk','tiempo_sk','estado_sk']:
        fact[col] = fact[col].astype(int)
    logger.info(f'fact_resultado_carrera: {len(fact):,} filas')
    return fact

fact = transform_fact(raw, dim_piloto, dim_constructor, dim_circuito, dim_tiempo, dim_estado)
fact.head()

## 6. LOAD — Carga en Aurora PostgreSQL

In [ ]:
def load_table(df, table_name, engine, schema='f1_dw'):
    full  = f'{schema}.{table_name}'
    cols  = ', '.join(df.columns)
    ph    = ', '.join([f':{c}' for c in df.columns])
    recs  = df.to_dict(orient='records')
    with engine.begin() as conn:
        conn.execute(text(f'TRUNCATE TABLE {full} RESTART IDENTITY CASCADE'))
        for i in range(0, len(recs), 1000):
            conn.execute(text(f'INSERT INTO {full} ({cols}) VALUES ({ph})'), recs[i:i+1000])
    logger.info(f'  OK {len(df):,} filas -> {full}')

def load_fact(df, engine, schema='f1_dw'):
    full  = f'{schema}.fact_resultado_carrera'
    cols  = ', '.join(df.columns)
    ph    = ', '.join([f':{c}' for c in df.columns])
    upd   = ', '.join([f'{c}=EXCLUDED.{c}' for c in df.columns if c not in ['piloto_sk','tiempo_sk']])
    sql   = text(f'INSERT INTO {full} ({cols}) VALUES ({ph}) ON CONFLICT (piloto_sk,tiempo_sk) DO UPDATE SET {upd}')
    recs  = df.to_dict(orient='records')
    with engine.begin() as conn:
        for i in range(0, len(recs), 1000):
            conn.execute(sql, recs[i:i+1000])
    logger.info(f'  OK {len(df):,} filas -> {full}')

load_table(dim_piloto,      'dim_piloto',      engine)
load_table(dim_constructor, 'dim_constructor', engine)
load_table(dim_circuito,    'dim_circuito',    engine)
load_table(dim_tiempo,      'dim_tiempo',      engine)
load_table(dim_estado,      'dim_estado',      engine)
load_fact(fact, engine)

## 7. VALIDATE — Validaciones post-carga

In [ ]:
def validate(raw, engine, schema='f1_dw'):
    errores = 0
    with engine.connect() as conn:
        print('\n Conteo BD vs CSV:')
        for tabla, src in [('dim_piloto','drivers'),('dim_constructor','constructors'),
            ('dim_circuito','circuits'),('dim_tiempo','races'),
            ('dim_estado','status'),('fact_resultado_carrera','results')]:
            real = conn.execute(text(f'SELECT COUNT(*) FROM {schema}.{tabla}')).scalar()
            esp  = len(raw.get(src, pd.DataFrame()))
            ok   = 'OK' if real==esp else 'DIFERENCIA'
            print(f'  {ok} {tabla:<35} BD:{real:>8,}  CSV:{esp:>8,}')
            if real!=esp: errores+=1
        print('\n FKs huerfanas en fact:')
        for fk,dim,pk in [('piloto_sk','dim_piloto','piloto_sk'),
            ('constructor_sk','dim_constructor','constructor_sk'),
            ('circuito_sk','dim_circuito','circuito_sk'),
            ('tiempo_sk','dim_tiempo','tiempo_sk'),
            ('estado_sk','dim_estado','estado_sk')]:
            h = conn.execute(text(f'SELECT COUNT(*) FROM {schema}.fact_resultado_carrera f '
                f'LEFT JOIN {schema}.{dim} d ON f.{fk}=d.{pk} WHERE d.{pk} IS NULL')).scalar()
            print(f'  {"OK" if h==0 else str(h)+" huerfanas"} fact.{fk} -> {dim}')
            if h>0: errores+=1
        print('\n Nulos en columnas criticas de fact:')
        for col in ['piloto_sk','constructor_sk','circuito_sk','tiempo_sk','estado_sk','puntos']:
            n = conn.execute(text(f'SELECT COUNT(*) FROM {schema}.fact_resultado_carrera WHERE {col} IS NULL')).scalar()
            print(f'  {"OK" if n==0 else str(n)+" nulos"} {col}')
            if n>0: errores+=1
    print(f'\nResultado: {"TODAS LAS VALIDACIONES PASARON" if errores==0 else str(errores)+" ERRORES"}')
    return errores==0

validate(raw, engine)

## 8. Resumen final de carga

In [ ]:
with engine.connect() as conn:
    print('Resumen final en f1_dw:\n')
    for t in ['dim_piloto','dim_constructor','dim_circuito','dim_tiempo','dim_estado','fact_resultado_carrera']:
        n = conn.execute(text(f'SELECT COUNT(*) FROM f1_dw.{t}')).scalar()
        print(f'  {t:<35} {n:>10,} filas')